# 🏭 Aula 21 — Implantação e Governança de IA**Disciplina:** IA Aplicada à Engenharia Química**Dataset:** shadow_test_composicao.csv — 30 dias de shadow test do soft-sensor---## A verdade dura"O modelo perfeito que nunca chegou à planta vale ZERO."ML é só ~10% do esforço; o resto é engenharia de implantação.

## As 4 Muralhas da Implantação1. **Qualidade de dados** — 6 meses de histórico; valores congelados; OPC-UA cai ~1%2. **Conectividade** — firewall; latência; edge (50ms) vs cloud (500ms)3. **Integração com legado** — PIMS que só aceita CSV4. **Governança** — aprovação, retreino, rollback, champion/challenger

## 3.1 — Exercício Guiado: Champion/Challenger

### Passo 1: carregar shadow test

In [ ]:
import pandas as pd, numpy as npURL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula21/shadow_test_composicao.csv"df = pd.read_csv(URL, parse_dates=['timestamp'])df.head()

### Passo 2: comparar MSE champion vs challenger

In [ ]:
mse_champ = np.mean((df['champion']-df['composicao_lab'])**2)mse_chal  = np.mean((df['challenger']-df['composicao_lab'])**2)eps = 0.10 * mse_champ     # margem de seguranca p/ nao promover por ruidoprint(f"MSE champion  = {mse_champ:.6f}")print(f"MSE challenger = {mse_chal:.6f}")print(f"Margem eps     = {eps:.6f} (10%)")promover = mse_chal < mse_champ - epsprint(f"Promover challenger?  {promover}")

### Passo 3: erro por dia (drift/promocao)

In [ ]:
df['err_champ'] = (df['champion']-df['composicao_lab']).abs()df['err_chal'] = (df['challenger']-df['composicao_lab']).abs()diario = df.groupby('dia')[['err_champ','err_chal']].mean()diario.plot(figsize=(10,4), marker='o')import matplotlib.pyplot as plt; plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Passo 4: plano de implantacao em 4 fases (preencha)

In [ ]:
# Fase | Acao | Risco | Mitigacao | Criterioplan = pd.DataFrame({  'Fase': [1,2,3,4],  'Acao': ['Off-line: validar 6 meses','Shadow: roda em paralelo',           'Assistida: operador valida','Autonoma: modelo atua'],  'Criterio_avancar': ['MAPE <= 2%','diferenca vs sensor <= 1.5% p/ 30d',                       '>=85% sugestoes aceitas','rollback < 5min; erro >3% reverte'],})plan

### Passo 5: ROI em 3 cenarios

In [ ]:
producao = 100_000   # t/anopreco = 800           # R$/tfaturamento = producao * precoperdas_base = 0.008   # 0.8% do faturamento (fração de perdas evitáveis)custo_impl = 96_000   # R$/ano (edge + retreino + equipe)for nome, red in [('pessimista',0.001),('realista',0.0025),('otimista',0.005)]:    economia = faturamento * perdas_base * (red/0.008)    roi = (economia - custo_impl)/custo_impl    payback = custo_impl/economia*12    print(f"{nome:<11} economia=R${economia:,.0f}/ano  ROI={roi*100:5.1f}%  payback={payback:.1f}meses")

> **Conclusao:** recomendar edge (latencia <200ms) mesmo com custo maior;> challenger promovido se MSE < champion - eps; ROI > 1 => viavel.

## 3.2 — Exercício em Grupo: Barreiras ReaisCada grupo defende 1 barreira e apresenta solução em 2 min:- (A) PIMS so aceita CSV  -> solução: micro-serviço que escreve CSV + monitora- (B) Só 2 meses de dados -> validar com metodos robustos + mais sensor- (C) Edge (12k, 50ms) vs Cloud (4k, 500ms) -> controle <200ms => edge- (D) Operadores resistentes -> shadow visível, treinamento, transparência

## Checklist de Governança- [ ] Dados auditados (6 meses, sem valores congelados)- [ ] Conectividade resolvida (firewall/edge)- [ ] Integração com legado (PIMS/CSV)- [ ] 4 fases definidas com critérios- [ ] ROI calculado (3 cenários)- [ ] Plano de rollback testado- [ ] Aprovação documentada- [ ] Champion/challenger em shadow mode